# Session 4 · Part 1 — Data types and DataFrames

Today is about the **practical side of working with data**: what form it arrives in, how to load it, how to spot when it's broken, and what to do when it's too big for Excel.
Analysing it properly is what your modules are for.

**Predict, then run.** Before running each cell in Part A, say out loud (or write down) what you think it will print.

## Part A · Why data types matter

In [ ]:
price = 350
price_text = "350"
print(type(price))
print(type(price_text))

In [ ]:
print(price + 50)
print(price_text + "50")     # predict first!

In [ ]:
print(price_text + 50)       # predict: what happens here?

Same-looking value, different **type**, completely different behaviour. Text is `str`, whole numbers are `int`, decimals are `float`.
You saw this in the Session 3 boss level: one `"350"` in a list of numbers crashed the whole calculation.
To convert: `int("350")`, `float("1250.75")`.

In [ ]:
print(int(price_text) + 50)
print(float("1250.75"))
print(float("1,250.75"))     # predict!

A comma is fine for humans and fatal for `float()`. Remember this, because you'll meet it again in a real file in a minute.

### Money and decimals

In [ ]:
print(0.1 + 0.2)
print(0.1 + 0.2 == 0.3)

Computers store decimals in binary, so tiny rounding errors creep in. For a chart that's harmless. For **money**, where every fils must reconcile, use `Decimal`, or round before comparing:

In [ ]:
from decimal import Decimal
print(Decimal("0.1") + Decimal("0.2"))
print(round(0.1 + 0.2, 2) == 0.3)

### Containers: list, tuple, dict
- **list** `[...]`: an ordered collection you can change, e.g. a series of prices
- **tuple** `(...)`: like a list but **can't be changed**, e.g. a fixed currency pair
- **dict** `{key: value}`: a lookup table, e.g. holdings in a portfolio

In [ ]:
prices = [101.5, 102.0, 99.8, 103.2]          # list
pair = ("USD", "AED")                         # tuple
portfolio = {"AAPL": 10, "MSFT": 5, "EMAAR": 200}   # dict

prices.append(104.0)
portfolio["TSLA"] = 3
print(prices)
print(pair[0], "->", pair[1])
print(portfolio["EMAAR"], "EMAAR shares")
print("NVDA" in portfolio)                    # a bool: True / False

## Part B · A DataFrame is a spreadsheet in Python

A **dict of lists** becomes a table. Keys become column names, lists become columns.

In [ ]:
import pandas as pd

holdings = pd.DataFrame({
    "ticker": ["AAPL", "MSFT", "EMAAR"],
    "shares": [10, 5, 200],
    "price":  [227.5, 415.0, 14.2],
})
holdings["value"] = holdings["shares"] * holdings["price"]   # a new column, no dragging formulas down
holdings

## Part C · Loading a real (messy) file

`data/transactions_sample.csv` is a bank-transactions export. Like most real exports, **it has problems**. Let's find them.

In [ ]:
df = pd.read_csv("data/transactions_sample.csv")
print(df.shape)       # (rows, columns)
df.head()

In [ ]:
df.info()

Look at the type of `amount_aed`. It should be a number, but pandas loaded it as **text** (`object` or `str`). That means at least one value in the column isn't a clean number. Let's find out which.

In [ ]:
df[df["amount_aed"].str.contains(",", na=False)].head()

The thousands separators (`1,250.00`). Remove the commas, then convert. `errors="coerce"` turns anything that still can't be converted into a missing value (`NaN`) instead of crashing.

In [ ]:
df["amount_aed"] = pd.to_numeric(df["amount_aed"].str.replace(",", ""), errors="coerce")
df["amount_aed"].describe()

### Problem 2 · Missing values

In [ ]:
df.isna().sum()

pandas automatically treated blank cells and `N/A` as missing. A transaction with no amount is useless here, so we drop those rows. Missing `country` is less serious, so we label it instead.

In [ ]:
df = df.dropna(subset=["amount_aed"])
df["country"] = df["country"].fillna("Unknown")
df.isna().sum()

### Problem 3 · The same category spelt different ways

In [ ]:
df["category"].value_counts()

In [ ]:
df["category"] = df["category"].str.strip().str.title()     # remove spaces, then Title Case
df["category"].value_counts()

### Problem 4 · Duplicate rows

In [ ]:
print("Duplicate rows:", df.duplicated().sum())
df = df.drop_duplicates()
print("Rows now:", len(df))

### Problem 5 · Dates stored as text

In [ ]:
df["date"] = pd.to_datetime(df["date"])
df.dtypes

## Part D · The everyday moves

Select columns → filter rows → sort → group. Once the data is clean, these few lines replace a lot of Excel clicking.

In [ ]:
big_travel = df[(df["category"] == "Travel") & (df["amount_aed"] > 2000)]
big_travel[["date", "customer_id", "amount_aed"]].sort_values("amount_aed", ascending=False).head()

In [ ]:
by_category = df.groupby("category")["amount_aed"].sum().sort_values(ascending=False)
by_category

In [ ]:
ax = by_category.plot(kind="barh", title="Total spend by category (AED)")
ax.invert_yaxis()

In [ ]:
df.to_csv("data/transactions_clean.csv", index=False)
print("Saved")

### Same data, other sources
Files don't only come as CSV. The loading line changes; everything after it stays the same.

In [ ]:
from_excel = pd.read_excel("data/transactions_sample.xlsx")     # needs openpyxl
print(from_excel.shape)

# Straight from a web link works too (if the wifi is up):
try:
    titanic = pd.read_csv("https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv")
    print(titanic.shape)
except Exception as e:
    print("No internet right now:", type(e).__name__)

## Your turn (pairs)

Use the cleaned `df`.
1. How many transactions were made through the **Online** channel?
2. Total spend per **country**, largest first.
3. Which **month** had the highest total spend? *(Hint: `df["date"].dt.month`)*

In [ ]:
# 1

In [ ]:
# 2

In [ ]:
# 3